# Experiment 1: productivity

In [1]:
import os
from pathlib import Path

from AUTOclui import AUTOCommands as ac
from AUTOclui import runAUTO as ra
from pyvirtualdisplay import Display

In [2]:
folder = Path.cwd()
os.chdir(folder)

model_name = 'common_model'
output_folder = folder / 'output_experiment_one_productivity'
output_folder.mkdir(exist_ok=True)

parameter_file = folder / 'experiment_parameters.dat'
parameter_file.write_text('0.0 0.0\n')

8

In [3]:
display = Display(visible=False, size=(1200, 900))
display.start()

In [4]:
runner = ra.runAUTO()

try:
    eq_forward = ac.run(e=model_name, c=model_name, runner=runner, NMX=4000, NPR=200)
    eq_backward = ac.run(DS='-', runner=runner, NMX=4000, NPR=200)
    eq = (eq_forward + eq_backward).relabel()
    ac.save(eq, 'eq')

    bp_curve = ac.run(
        eq('BP1'),
        ICP=[28, 29],
        ISW=2,
        DS=1.0e-3,
        DSMIN=1.0e-5,
        DSMAX=5.0e-3,
        NMX=8000,
        NPR=400,
        UZSTOP={28: [-0.25, 0.5], 29: [0.0, 1.0]},
        runner=runner,
    ).relabel()
    ac.save(bp_curve, 'bp_curve')

    lp_curve = ac.run(
        eq('LP1'),
        ICP=[28, 29],
        ISW=2,
        DS=1.0e-3,
        DSMIN=1.0e-5,
        DSMAX=5.0e-3,
        NMX=8000,
        NPR=400,
        UZSTOP={28: [-0.25, 0.5], 29: [0.0, 1.0]},
        runner=runner,
    ).relabel()
    ac.save(lp_curve, 'lp_curve')

    codim2 = (bp_curve + lp_curve).relabel()
    ac.save(codim2, 'codim2')
finally:
    runner.config(clean=True)
    ac.clean()

gfortran -g -fopenmp -O -c common_model.f90 -o common_model.o
gfortran -g -fopenmp -O common_model.o -o common_model.exe /auto/lib/*.o
Starting common_model ...

  BR    PT  TY  LAB       mu         L2-NORM          PL            FL            JL            PP            FP            JP      
   1     1  EP    1   0.00000E+00   9.42809E+00   0.00000E+00   6.66667E+00   0.00000E+00   0.00000E+00   6.66667E+00   0.00000E+00
   1     7  BP    2   2.24490E-02   9.42809E+00   0.00000E+00   6.66667E+00   0.00000E+00   0.00000E+00   6.66667E+00   0.00000E+00
   1    18  UZ    3   5.00000E-01   9.42809E+00   0.00000E+00   6.66667E+00   0.00000E+00   0.00000E+00   6.66667E+00   0.00000E+00

  BR    PT  TY  LAB       mu         L2-NORM          PL            FL            JL            PP            FP            JP      
   2    53  LP    4   3.72113E-02   7.44459E+00   3.97058E-01   5.24075E+00   2.96323E-01   3.97058E-01   5.24075E+00   2.96323E-01
   2   147  MX    5  -5.73137E-02   5.93272

In [5]:
p = ac.plot('eq', hide=True)
p.config(
    stability=True,
    grid=False,
    bifurcation_x=['mu'],
    bifurcation_y=['PL'],
    xlabel='mu',
    ylabel='PL',
    title='',
    minx=0.0,
    maxx=0.1,
)
p.savefig(str(output_folder / 'experiment_one_productivity_1d.png'))
p.savefig(str(output_folder / 'experiment_one_productivity_1d.svg'))

Created plot


In [6]:
p = ac.plot('codim2', hide=True)
p.config(
    grid=False,
    bifurcation_x=['deltar'],
    bifurcation_y=['mu'],
    xlabel='deltar',
    ylabel='mu',
    title='',
    minx=0.0,
    maxx=1.0,
    miny=0.0,
    maxy=0.1,
)
p.savefig(str(output_folder / 'experiment_one_productivity_2d.png'))
p.savefig(str(output_folder / 'experiment_one_productivity_2d.svg'))

Created plot


In [7]:
display.stop()
parameter_file.unlink(missing_ok=True)
ac.delete('eq')
ac.delete('bp_curve')
ac.delete('lp_curve')
ac.delete('codim2')

Deleting b.eq ... done
Deleting s.eq ... done
Deleting d.eq ... done
Deleting b.bp_curve ... done
Deleting s.bp_curve ... done
Deleting d.bp_curve ... done
Deleting b.lp_curve ... done
Deleting s.lp_curve ... done
Deleting d.lp_curve ... done
Deleting b.codim2 ... done
Deleting s.codim2 ... done
Deleting d.codim2 ... done
